In [ ]:
import sys
print(sys.executable)
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
import scipy
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Dropout
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, BatchNormalization, MaxPooling2D,
                                       GlobalAveragePooling2D, Dense, Dropout)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [4]:
# Load the training dataset
train_ds = keras.utils.image_dataset_from_directory(
    directory='Dataset/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256, 256)
)

# Load the validation dataset
validation_ds = keras.utils.image_dataset_from_directory(
    directory='Dataset/test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256, 256)
)

print("==========================")
print("Dataset loaded sucessful")
print("==========================")

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Dataset loaded sucessful


In [5]:
# Normalize pixel values from [0, 255] to [0, 1]
def process(image, label):
    image = tf.cast(image / 255., tf.float32)
    return image, label

train_ds = train_ds.map(process)
validation_ds = validation_ds.map(process)

val_datagen = ImageDataGenerator(rescale=1./255)  # no augmentation on validation!

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)  # no augmentation for validation

In [7]:
train_generator = train_datagen.flow_from_directory(
    'dataset/train',              # change to your train folder path
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    'dataset/test',                # change to your validation folder path
    target_size=(256, 256),
    batch_size=32,
    class_mode='binary'
)

Found 20000 images belonging to 2 classes.
Found 5000 images belonging to 2 classes.


In [8]:
# ---------------------------
# 2. Model Architecture
# ---------------------------
model = Sequential()

# 1st Convolutional Block
model.add(Conv2D(32, kernel_size=(3, 3), padding='valid', activation='relu',
                  kernel_regularizer=l2(1e-4), input_shape=(256, 256, 3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

# 2nd Convolutional Block
model.add(Conv2D(64, kernel_size=(3, 3), padding='valid', activation='relu',
                  kernel_regularizer=l2(1e-4)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

# 3rd Convolutional Block
model.add(Conv2D(128, kernel_size=(3, 3), padding='valid', activation='relu',
                  kernel_regularizer=l2(1e-4)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

# Global Average Pooling instead of Flatten (fewer params, less overfitting)
model.add(GlobalAveragePooling2D())

# Fully Connected Layers
model.add(Dense(128, activation='relu', kernel_regularizer=l2(1e-4)))
model.add(Dropout(0.4))

model.add(Dense(64, activation='relu', kernel_regularizer=l2(1e-4)))
model.add(Dropout(0.3))

# Output Layer for Binary Classification
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 254, 254, 32)      896       
                                                                 
 batch_normalization (BatchN  (None, 254, 254, 32)     128       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2D  (None, 127, 127, 32)     0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 125, 125, 64)      18496     
                                                                 
 batch_normalization_1 (Batc  (None, 125, 125, 64)     256       
 hNormalization)                                                 
                                                        

In [9]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

In [10]:
import sys
print("Python:", sys.executable)

import numpy
print("NumPy:", numpy.__version__)

import scipy
print("SciPy:", scipy.__version__)

import tensorflow as tf
print("TF:", tf.__version__)

Python: d:\STUDY MAT\Deep Learning\CAT vs DOG\.venv\Scripts\python.exe
NumPy: 1.26.4
SciPy: 1.10.1
TF: 2.10.0


In [12]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)


Epoch 1/50
625/625 [==============================] - 214s 343ms/step - loss: 0.6920 - accuracy: 0.6033 - val_loss: 0.7069 - val_accuracy: 0.5928 - lr: 0.0010
Epoch 2/50
625/625 [==============================] - 178s 284ms/step - loss: 0.6510 - accuracy: 0.6538 - val_loss: 0.7054 - val_accuracy: 0.5546 - lr: 0.0010
Epoch 3/50
625/625 [==============================] - 181s 290ms/step - loss: 0.6155 - accuracy: 0.6895 - val_loss: 0.6285 - val_accuracy: 0.6816 - lr: 0.0010
Epoch 4/50
625/625 [==============================] - 179s 287ms/step - loss: 0.5899 - accuracy: 0.7117 - val_loss: 0.6464 - val_accuracy: 0.6510 - lr: 0.0010
Epoch 5/50
625/625 [==============================] - 182s 291ms/step - loss: 0.5733 - accuracy: 0.7273 - val_loss: 0.6528 - val_accuracy: 0.6962 - lr: 0.0010
Epoch 6/50
625/625 [==============================] - 182s 291ms/step - loss: 0.5576 - accuracy: 0.7371 - val_loss: 0.6955 - val_accuracy: 0.6716 - lr: 0.0010
Epoch 7/50
625/625 [==========================

In [13]:
model.save("model.keras")

In [14]:
# ==========================================
# 5. MODEL COMPILATION & TRAINING
# ==========================================
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Execute training across 10 epochs
history = model.fit(train_ds, epochs=10, validation_data=validation_ds)

print("=========================")
print("Model train sussesfull!!")
print("=========================")

model.save("model.keras")

Epoch 1/10


 56/625 [=>............................] - ETA: 1:32 - loss: 0.6039 - accuracy: 0.7126

KeyboardInterrupt: 

In [ ]:
# ==========================================
# 6. EVALUATION (LOSS & ACCURACY PLOTS)
# ==========================================
# Plot Accuracy
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], color='red', label='train')
plt.plot(history.history['val_accuracy'], color='blue', label='validation')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], color='red', label='train')
plt.plot(history.history['val_loss'], color='blue', label='validation')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

print("END")

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

In [ ]:
MODEL_PATH = 'model.h5'  # <-- change this if your saved model has a different name/path
 
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model file not found at '{MODEL_PATH}'. "
        f"Make sure you saved it after training with model.save('{MODEL_PATH}')."
    )
 
model = load_model(MODEL_PATH)
print(f"Loaded model from {MODEL_PATH}")

In [ ]:
# ---------------------------------------------------------
# 2. Test on a single image
# ---------------------------------------------------------
image_path = 'premium_photo-1666777247416-ee7a95235559.avif'
 
if os.path.exists(image_path):
    test_img = cv2.imread(image_path)
 
    if test_img is None:
        # cv2.imread returns None (not an exception) if it can't decode the file.
        # AVIF in particular isn't supported by all OpenCV builds.
        raise ValueError(
            f"cv2.imread couldn't decode '{image_path}'. "
            f"If this is an .avif file, your OpenCV build may not support it — "
            f"try converting it to .jpg/.png first, or install 'opencv-python' "
            f"with AVIF support / use Pillow instead to load it."
        )
 
    # Convert BGR (OpenCV default) to RGB for correct matplotlib plotting
    test_img_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
    plt.imshow(test_img_rgb)
    plt.axis('off')
    plt.show()
 
    # Preprocess the image to fit model expectations
    test_img_resized = cv2.resize(test_img, (256, 256))
    test_input = test_img_resized.reshape((1, 256, 256, 3))
 
    # Scale image pixels if the training pipeline scales them
    test_input = test_input / 255.0
 
    # Prediction
    prediction = model.predict(test_input)
    print(f"Raw Prediction Value: {prediction[0][0]}")
 
    if prediction[0][0] < 0.5:
        print("Prediction: It's a CAT!")
    else:
        print("Prediction: It's a DOG!")
else:
    print(f"Image not found at {image_path}. Please check your path.")
 
